# Rossmann Store Sales Forecasting

**NextHikes IT Solutions** &middot; Machine Learning Engineering Project for Rossmann Pharmaceuticals

This notebook is the analysis-and-modelling companion to the project's `src/` package. Every
transformation used here (cleaning, feature engineering, the sklearn pipeline, the loss
function, the LSTM) is implemented once in `src/` and imported, not duplicated in-notebook —
so the exact code that runs here is the same code the Flask app and the training scripts use.

**Contents**
1. Setup and data loading
2. Task 1 — Exploratory analysis of customer purchasing behaviour
3. Task 2.1–2.2 — Feature engineering and the sklearn pipeline
4. Task 2.3 — Loss function (RMSPE) and why it was chosen
5. Task 2.4–2.5 — Model comparison, feature importance, confidence intervals, serialization
6. Task 2.6 — LSTM deep-learning forecaster
7. Task 2.7 — MLflow tracking
8. Task 3 — Serving predictions (Flask app) — summary and screenshots
9. Conclusions


## 1. Setup and data loading

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import viz
from src.cleaning import clean_dataset, missing_value_report, outlier_report
from src.data_loader import load_dataset
from src.features import build_holiday_calendar, engineer_features
from src.logger import get_logger

viz.apply_theme()
%matplotlib inline
pd.set_option("display.max_columns", 60)

logger = get_logger("notebook")


In [ ]:
train_raw = load_dataset("train")
test_raw = load_dataset("test")

print(f"train: {train_raw.shape}, {train_raw.Date.min().date()} -> {train_raw.Date.max().date()}")
print(f"test : {test_raw.shape}, {test_raw.Date.min().date()} -> {test_raw.Date.max().date()}")
print(f"stores: train={train_raw.Store.nunique()}, test={test_raw.Store.nunique()}")
train_raw.head()


The three Kaggle files (`train.csv`, `test.csv`, `store.csv`) are joined on `Store`. The
project also pulls in three locality-related extras from the fast.ai mirror of this
dataset — `store_states.csv`, `weather.csv` and `googletrend.csv` — to give the "locality"
factor the brief calls out its own features (state, daily weather, weekly search-interest
trend). See `src/data_loader.py` for the join logic, including two data-format traps that
had to be handled explicitly: the Google-trend weeks run Sunday-to-Saturday (not the pandas
default Monday), and `store_states.csv` labels Bremen/Lower-Saxony jointly as `"HB,NI"`
while the trend file only publishes a series for `"NI"`.


In [ ]:
report = missing_value_report(train_raw)
print("Missing values before cleaning:")
report


## 2. Task 1 — Exploration of Customer Purchasing Behaviour

Cleaning is implemented as sklearn-compatible transformers (`src/cleaning.py`) so the exact
same logic runs in training and at serving time. Two decisions worth stating up front:

* **`CompetitionDistance` NaN does not mean "distance zero."** It means no competitor is on
  record, which behaves like a *very distant* competitor. It is imputed with a large
  sentinel plus an explicit `HasCompetition` flag, never with the mean.
* **Closed-store days (`Open == 0`) are dropped from training**, not learned. They always
  have `Sales == 0`; that is a deterministic business rule (`Sales = 0` whenever `Open = 0`),
  not something worth spending model capacity to rediscover.


In [ ]:
clean = clean_dataset(train_raw, outlier_strategy="flag", for_training=True)
calendar = build_holiday_calendar(train_raw, test_raw)
clean, _ = engineer_features(clean, holiday_calendar=calendar)

print(f"trading rows after cleaning: {len(clean):,} of {len(train_raw):,} raw rows")
print(f"engineered columns: {clean.shape[1]}")


### Running the full analysis suite

`src/eda.py` implements one function per question — each returns a `Finding` bundling the
chart, the evidence table and the written insight. `run_all` executes all 14 (11 from the
brief's question list, 3 of our own) and also writes `reports/eda_findings.md` and the PNGs
under `reports/figures/`. The cell below reruns them inline so the charts render in the
notebook; the full write-up with every question, table and insight is in
[`reports/eda_findings.md`](../reports/eda_findings.md).


In [ ]:
from src import eda

findings = eda.run_all(clean, train_raw, test_raw, close_figures=False)
print(f"{len(findings)} findings generated")


In [ ]:
# Display every finding inline: question, chart, evidence table, insight.
for f in findings:
    print("=" * 100)
    print(f"Q: {f.question}")
    print("=" * 100)
    if f.figure is not None:
        display(f.figure)
    if f.table is not None:
        display(f.table)
    print(f"\nINSIGHT: {f.insight}\n")


## 3. Task 2.1–2.2 — Feature Engineering and the sklearn Pipeline

`src/features.py` extracts every feature the brief asks for and several extras:

* **Calendar**: weekday, weekend, ISO week, day-of-year, quarter, cyclical sin/cos
  encodings (so December sits next to January instead of 11 units away)
* **Month phase**: beginning / mid / end of month, plus days-to-month-end and an
  `IsPayWeek` flag (German wages land at month end — a real demand driver, see Q3/Q14)
* **Holidays**: signed `DaysToNextHoliday` / `DaysAfterLastHoliday`, built from a calendar
  that unions train *and* test holiday dates so forecast rows near the test boundary still
  know about an upcoming holiday
* **Competition**: months since the nearest competitor opened, a log-distance transform,
  and an `IsCityCentre` proxy — Q10 showed raw distance mostly measures urban density, not
  competitive pressure
* **Promo2**: `PromoInterval` (e.g. `"Feb,May,Aug,Nov"`) decoded into a genuine per-row
  `IsPromo2Active` flag, checking both the month-list membership and the join date
* **Store history**: per-store mean/median/std sales, per-store-weekday and per-store-month
  means, and basket size (`Sales / Customers`) — Q12 showed store identity alone explains
  ~60% of total sales variance, making these the single highest-value feature family

All of it is assembled into one `sklearn.pipeline.Pipeline` in `src/pipeline.py`, so cleaning,
feature engineering, scaling/encoding and the model are one fittable, picklable object. That
is what lets the Flask app in `app/predictor.py` call `pipeline.predict(raw_dataframe)`
directly with no preprocessing code duplicated between training and serving.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

from src.pipeline import fit_full_pipeline, get_feature_names
from src.config import RANDOM_STATE, VALIDATION_WEEKS

# Chronological split: the final 6 weeks held out, mirroring the brief's forecast horizon.
# A random split would leak each store's own future statistics into its own past.
cutoff = clean["Date"].max() - pd.Timedelta(weeks=VALIDATION_WEEKS)
tr_demo = clean[clean["Date"] <= cutoff]
va_demo = clean[clean["Date"] > cutoff]
print(f"demo split at {cutoff.date()}: train {len(tr_demo):,} / valid {len(va_demo):,}")

# A modest demo model (small forest, single random store sample) so this cell runs in
# seconds; the full-scale comparison (all 1,115 stores, tuned RandomForest + LightGBM) is
# run via `scripts/train_ml.py` and tracked in MLflow -- see Section 5 for those numbers.
demo_stores = np.random.RandomState(RANDOM_STATE).choice(clean.Store.unique(), 80, replace=False)
tr_small = tr_demo[tr_demo.Store.isin(demo_stores)]
va_small = va_demo[va_demo.Store.isin(demo_stores)]

y_tr_small = np.log1p(tr_small["Sales"].values)
demo_pipe = fit_full_pipeline(
    tr_small, y_tr_small,
    model=RandomForestRegressor(n_estimators=60, min_samples_leaf=2, max_features="sqrt",
                                 n_jobs=-1, random_state=RANDOM_STATE),
    holiday_calendar=calendar,
)
print(f"pipeline fitted: {len(get_feature_names(demo_pipe))} encoded features")


## 4. Task 2.3 — Loss Function: RMSPE

**Chosen metric: Root Mean Square Percentage Error.**

$$\text{RMSPE} = \sqrt{\frac{1}{n}\sum_i \left(\frac{y_i - \hat{y}_i}{y_i}\right)^2}$$

**Why this one, over plain RMSE:**

1. **Scale invariance across a heterogeneous estate.** Q12 showed store size varies
   sixfold. Under RMSE a 10% miss on a flagship store contributes ~100x more squared error
   than the same 10% miss on a small store, so the optimiser would effectively ignore small
   stores — but the finance team needs a usable forecast for *every* store.
2. **Direct interpretability.** "Our forecasts are within 12% on average" is something a
   finance analyst can act on immediately; "RMSE is 840" requires knowing each store's
   baseline first.
3. **It is the metric the original Kaggle competition scored on**, so results are
   comparable with published benchmarks (top solutions ≈ 0.10).

Rows with `Sales == 0` are masked out of the ratio (undefined at zero) — this never bites in
practice because closed days are excluded from training and handled by the deterministic
`Open = 0 -> Sales = 0` rule instead. See `src/metrics.py` for the full implementation and
the supporting MAE/MAPE/R² bundle reported alongside it.


In [ ]:
from src.metrics import evaluate

pred_demo = np.expm1(demo_pipe.predict(va_small))
demo_metrics = evaluate(va_small["Sales"].values, pred_demo, prefix="valid")
pd.Series(demo_metrics)


## 5. Task 2.4–2.5 — Model Comparison, Feature Importance, Confidence Intervals, Serialization

The full-scale comparison trains two models behind the identical pipeline on all 1,115
stores (`scripts/train_ml.py`), differing only in the final estimator:

* **RandomForestRegressor** — the brief's suggested starting point
* **LightGBM** — gradient-boosted trees, the "innovative approach" upgrade

Both train on `log1p(Sales)` (right-skewed target, see Q14) and are scored back on the real
Sales scale. Results are logged to MLflow (Section 7) and the fitted pipelines are
serialized with a timestamp (`src/serialize.py`, format `rf_10-08-2020-16-32-31-00.pkl`) so
each day's retrain is independently addressable.


In [ ]:
import json
from src.config import MODELS_DIR

rows = []
for meta_path in sorted(MODELS_DIR.glob("*.json")):
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        continue
    m = meta.get("metrics", {})
    if m:
        rows.append({
            "model": meta.get("model_name"),
            "file": meta.get("file"),
            "timestamp": meta.get("timestamp"),
            "rmspe": m.get("valid_rmspe"),
            "rmse": m.get("valid_rmse"),
            "mae": m.get("valid_mae"),
            "mape": m.get("valid_mape"),
            "r2": m.get("valid_r2"),
            "fit_seconds": meta.get("fit_seconds"),
        })

comparison = pd.DataFrame(rows).sort_values("rmspe") if rows else pd.DataFrame()
comparison


**Post-prediction analysis (`src/confidence.py`):**

* **Feature importance** is reported by *permutation importance* (drop in validation score
  when a column is shuffled), not the Random Forest's built-in `feature_importances_`,
  which is biased toward high-cardinality numeric columns.
* **Confidence intervals** reuse the Random Forest's own trees: each of the ~200 trees
  gives an independent prediction for a row, and the 2.5th/97.5th percentile across trees
  is a genuinely free 95% interval — no second model needed. This is read as "how much the
  forest's trees disagree," not a full predictive interval, but it does the practical job:
  wider spreads flag rows (new stores, unusual promo combinations) the forecast should be
  trusted less on.


In [ ]:
import matplotlib.image as mpimg
from src.config import REPORTS_DIR

for model_name in ["random_forest", "lightgbm"]:
    path = REPORTS_DIR / "figures" / f"feature_importance_{model_name}.png"
    if path.exists():
        plt.figure(figsize=(10, 7))
        plt.imshow(mpimg.imread(path))
        plt.axis("off")
        plt.title(f"{model_name}: permutation importance")
        plt.show()


In [ ]:
interval_path = REPORTS_DIR / "prediction_intervals_random_forest.csv"
if interval_path.exists():
    intervals = pd.read_csv(interval_path)
    coverage = ((intervals["actual"] >= intervals["lower"]) & (intervals["actual"] <= intervals["upper"])).mean()
    print(f"empirical coverage of the 95% tree-spread interval: {coverage:.1%}")
    intervals.head(10)


## 6. Task 2.6 — LSTM Deep-Learning Forecaster

Implemented in `src/lstm_model.py`, following the brief's seven steps exactly, and run at
full scale via `scripts/train_lstm.py`:

1. **Isolate a time series** — national daily sales (sum across all open stores) for the
   stationarity/ACF diagnostics, since one clean series is easiest to read
2. **Test stationarity** — Augmented Dickey-Fuller test
3. **Difference if needed** — first-differenced only if the ADF test fails to reject a unit
   root
4. **ACF / PACF** — used to pick the LSTM lookback window (the weekly cycle at lag ≈7 is the
   dominant structure in retail sales)
5. **Sliding window → supervised learning** — built **per store**, then stacked, so the
   network trains on far more than the ~900 points a single aggregate series would give
6. **Scale to (-1, 1)** — per store, with its own `MinMaxScaler`, because store size varies
   sixfold (Q12) and a single global scale would let the largest stores dominate the loss
7. **Two-layer LSTM regressor** (50 then 25 units) — kept shallow per the brief so it trains
   comfortably on a CPU or a free Colab instance

See [`reports/figures/lstm_stationarity_acf_pacf.png`](../reports/figures/lstm_stationarity_acf_pacf.png)
and [`reports/figures/lstm_training_and_fit.png`](../reports/figures/lstm_training_and_fit.png)
for the diagnostic charts, and the MLflow `lstm` run (Section 7) for the full metrics.


In [ ]:
for name in ["lstm_stationarity_acf_pacf", "lstm_training_and_fit"]:
    path = REPORTS_DIR / "figures" / f"{name}.png"
    if path.exists():
        plt.figure(figsize=(12, 5))
        plt.imshow(mpimg.imread(path))
        plt.axis("off")
        plt.show()


## 7. Task 2.7 — MLflow Tracking

`src/mlflow_utils.py` points MLflow at a project-local SQLite store under `mlruns/`
(MLflow 3.x retired the plain file-store backend), so no external server is needed. Every
training run (`random_forest`, `lightgbm`, `lstm`) logs its parameters, the full metric
bundle, feature-importance artefacts, and the serialized model path. Open the dashboard
with:

```bash
mlflow ui --backend-store-uri "sqlite:///mlruns/mlflow.db" --port 5000
```

Screenshots of the dashboard showing multiple runs/model versions are included in the
interim submission per the brief's requirement.


## 8. Task 3 — Serving Predictions (Flask App)

The `app/` package serves the trained pipeline behind a small Flask dashboard:

* **`GET /`** — pick a store, upload a CSV of forecast dates (`Date`, `IsHoliday`,
  `IsWeekend`, `IsPromo`, `SchoolHoliday` — missing columns default to 0)
* **`POST /predict`** — runs `app/predictor.py::predict_for_store`, which calls the *same*
  serialized pipeline used here, with no preprocessing logic duplicated
* Shows a chart of predicted sales (with the Random Forest's confidence band where
  available) and an estimated customer count, derived from the store's own learned
  sales-per-customer ratio
* **`GET /download/<token>`** — downloads the shown predictions as CSV
* **`GET /api/predict`** — a JSON equivalent for programmatic use

Run locally with:

```bash
python app/app.py
```

and open `http://127.0.0.1:5000`. See the project README for deployment notes (Render /
Hugging Face Spaces — Heroku's free tier no longer exists).


## 9. Conclusions

* Sales forecasting for this estate is dominated by **store identity** (~60% of variance,
  Q12) and **calendar structure** (day-of-week, month phase, proximity to a holiday) far
  more than by the attributes usually assumed to matter most, like competitor distance —
  which Q10 showed is mostly a proxy for urban density once locality is controlled for.
* The daily **Promo** flag delivers a genuine, roughly balanced lift in both footfall and
  basket size (Q5); the long-running **Promo2** shows no measurable effect and is worth
  flagging to the finance team as a candidate for review.
* Promo response is highly uneven across stores (Q6): concentrating spend on the top
  quartile of responders while withdrawing it from the bottom quartile is the concrete,
  actionable recommendation from this analysis — with the caveat that it is observational,
  not a randomised test.
* RMSPE was chosen as the loss specifically because of the sixfold store-size spread — it
  is the metric that treats a small store's forecast as seriously as a flagship's, which is
  what the finance team's brief actually asks for.
* Gradient-boosted trees (LightGBM) are compared directly against the brief's suggested
  Random Forest baseline behind an identical pipeline; see Section 5 for which one won on
  this validation split.
* The LSTM (Section 6) demonstrates the full classical-to-deep-learning time series
  workflow the brief asks for, trained across many stores' windowed histories rather than
  a single aggregate series, so it reflects genuine per-store dynamics rather than only the
  network-wide trend.
